## **Nugen Intelligence**
<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="200"/>

Domain-aligned foundational models at industry leading speeds and zero-data retention! To learn more, visit [Nugen](https://docs.nugen.in/introduction)

### **Vision Alignmeent with the Nugen API**
 
This cookbook demonstrates how to create a Vision Alignment Project using the Nugen API. You'll learn how to upload an image dataset, automatically generate benchmark questions, train an aligned vision model, monitor training progress, and finally perform inference using the aligned model.

The notebook explains each step in a simple, sequential manner so that you can easily reproduce the complete workflow.

### **Dataset Preparation**

Before creating a Vision Alignment project, your image dataset must be preprocessed.

- Convert every image to a **Base64-encoded string**. 
  Eample-{"image": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAKAAAAAoCAIAAAD2TmbPAAAL}
- Create a **`.jsonl`** file containing one JSON object per line.
- Each JSON object should include the Base64-encoded image and the corresponding annotation or metadata required for training.
- Save the file with the `.jsonl` extension.
- Use this `.jsonl` file when uploading the dataset for Vision Alignment.

> **Note:** Vision Alignment accepts image datasets in Base64 format stored in a `.jsonl` file. Raw image files (such as `.jpg` or `.png`) must be converted before starting the alignment process.

### **Workflow**                                                                                  
The cookbook covers the following steps:

1 Upload a vision dataset (.jsonl).
2 Retrieve document details.
3 Generate benchmark questions from the uploaded dataset.
4 Create a Vision Alignment project.
5 Monitor alignment training status.
6 Run inference using the aligned vision model.

### **Dataset Split**

After uploading your dataset, 15% of the uploaded data will automatically be used to generate benchmark questions for evaluating the aligned model. The remaining data is used during the alignment process.

Uploaded Dataset
      - 85% → Alignment Training
      - 15% → Benchmark Generation

### **Step 1**

**Install the required Python packages**

In [1]:
!pip install --quiet requests pandas python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


**Import Required Libraries**

In [2]:
import os 
import requests
import pandas as pd
import json
import time
import base64
from dotenv import load_dotenv
load_dotenv()

True

**Set up the Nugen API Client**

To read more about Nugen API and access free API keys, you can visit [Nugen Dashboard](https://platform.nugen.in/)

**Configure the API Client**

Set your Nugen API key.

In [3]:
api_key = os.getenv("NUGEN_API_KEY")

In [4]:
headers = {"Authorization": f"Bearer {api_key}"}

Here, we define the API base URL and your API key. Replace <--nugen api key--> with your actual key to authenticate your requests to the Nugen API. The MODEL variable specifies the model we will use for generating the routines.

In [5]:
BASE_URL = "https://api.nugen.in"

### **Step 2**

**Upload the Vision Dataset**

Upload a JSONL dataset

In [6]:
url = f"{BASE_URL}/api/v3/documents/create"

In [ ]:
with open("vision_dataset.jsonl", "rb") as f:
    files = {
        "files": ("vision_dataset.jsonl", f, "application/json")
    }

    data = {
        "categories": "image"
    }

    document_response = requests.post(
        url,
        headers=headers,
        data=data,
        files=files
    )
print(document_response.text)


{"documents":["doc_01M1B766NV9DGJJ"]}


**Get status of Document uploaded**

Once the upload is complete, retrieve the document ID

The status of a document upload task, at the address a status lives at.

In [8]:
response_data = json.loads(document_response.text) 

id = response_data["documents"][0]


In [9]:
document_status_url = f"{BASE_URL}/api/v3/documents/{id}/status"

In [10]:
document_status_response = requests.get(document_status_url, headers=headers)

print(document_status_response.text)

{"status":"READY","document_id":"doc_01M1B766NV9DGJJ"}


**Generate Benchmark Questions**

Generate evaluation questions from the uploaded dataset.

Note: Benchmark questions are generated using 15% of the uploaded dataset

In [11]:
response_data = json.loads(document_status_response.text) 

document_id = response_data["document_id"]

In [ ]:
generate_benchmark_url = f"{BASE_URL}/api/v3/benchmarks/create"

In [13]:
payload = {
    "documents": [document_id],
    "num_questions": 20
}

benchmark_response = requests.post(generate_benchmark_url, json=payload, headers=headers)

print(benchmark_response.text)

{"benchmark_id":"benchmark_01M1B7BVSSVZ966","status":"PROCESSING"}


**Get status Benchmark Generation**

Check whether benchmark generation has completed.

In [17]:
response_data = json.loads(benchmark_response.text)

benchmark_id = response_data["benchmark_id"]

In [18]:
benchmark_status_url = f"{BASE_URL}/api/v3/benchmarks/{benchmark_id}/status"

In [19]:
response = requests.get(benchmark_status_url, headers=headers)

print(response.text)

{"benchmark_id":"benchmark_01M1B7BVSSVZ966","benchmark_name":"generated_benchmark_01M1B7BVSSVZ966","status":"READY","start_time":"2026-08-31T06:16:04.649615","end_time":"2026-08-31T06:16:05.670383"}


**Create a Vision Alignment Project**

**Example base model:**

qwen2-vl-2b-instruct

In [20]:
alignment_url = f"{BASE_URL}/api/v3/alignment-projects/create"

In [ ]:
payload = {
    "name": "My Vision Alignment ",
    "base_model": "qwen2-vl-2b-instruct",
    "document_ids": [document_id],
    "workflow_id": "workflow-abc123",
    "benchmark_id": benchmark_id,
    "description": "This project aims to align the model for better vision alignment."
}

print(payload)
alignment_response = requests.post(alignment_url, json=payload, headers=headers)

print(alignment_response.text)

{'name': 'Ak-31-august-My Vision Alignment ', 'base_model': 'qwen2-vl-2b-instruct', 'document_ids': ['doc_01M1B766NV9DGJJ'], 'workflow_id': 'workflow-abc123', 'benchmark_id': 'benchmark_01M1B7BVSSVZ966', 'description': 'This project aims to align the model for better vision alignment.'}
{"alignment_id":"alignment_01M1B7G9YKBZQFN","status":"PROCESSING"}


**Check Alignment Status**

Track the alignment status.

In [67]:
response_data = json.loads(alignment_response.text)

alignment_id = response_data["alignment_id"]

In [68]:
alignment_status_url = f"{BASE_URL}/api/v3/alignment-projects/{alignment_id}/status"

In [69]:
while True:
    alignment_status_response = requests.get(alignment_status_url, headers=headers)
    alignment_status_response.raise_for_status()
    print(alignment_status_response.text)
    data = alignment_status_response.json()
    status = data["status"]

    print(f"Current status of Alignment: {status}")

    if status == "READY":
        print("Alignment completed.")
        break

    if status == "FAILED":
        raise Exception("Alignment failed.")
    time.sleep(10)

{"alignment_id":"alignment_01M1B7G9YKBZQFN","status":"READY","queue_position":null,"degraded":null,"stage_failures":null,"stop_requested_at":null,"error":null,"updated_at":"2026-08-31 06:27:20.972568"}
Current status of Alignment: READY
Alignment completed.


### **List Aligned Model**

Retrieve all domain-aligned models for the authenticated user.

In [95]:
list_aligned_model_url= f"{BASE_URL}/api/v3/models/aligned"

In [96]:
list_aligned_model_response = requests.get(list_aligned_model_url, headers=headers)

print(list_aligned_model_response.text)

{"domain_aligned_models":[{"id":"ak-27-juy-prod-alignment-projectalignment-01kyhc5j6tyc455","alignment_id":"alignment_01KYHC5J6TYC455","name":"Ak-27-juy-prod-Alignment-Projectalignment_01KYHC5J6TYC455","base_model":"qwen-v2p5-0p5b-instruct","alignment_project":"Ak-27-juy-prod-Alignment-Project","created_date":"2026-07-27 08:52:20.071081","status":"READY","deployment_status":"READY","endpoint":"","creator":"abhishekkhodke21+101prod@gmail.com","usage_count":0,"last_used":null,"performance_metrics":{"accuracy":0.0,"uncertainty":0.0,"domain_violations":0,"average_response_time":0.0},"downloadable":false,"evaluation_data":{"evaluation_id":"eval_01KYHCD7V7HVQFH","status":"READY","metrics":{},"raw_answers_count":40,"completed_at":"2026-07-27T09:01:29.107849","created_at":"2026-07-27T08:52:33.770370","method":"eval-compare","baseline_model_id":"qwen-v2p5-0p5b-instruct","base_model":{"metrics":{"answer_relevance_max":1.0,"answer_relevance_min":0.0,"answer_relevance_std":0.4777813307361433,"answ

### **Get Model**

Get one aligned model by ID.

In [ ]:
response_data = json.loads(list_aligned_model_response.text)

model_id = response_data["domain_aligned_models"][0]["id"]

model_ak-31-august-my_vision_alignment__alignment_01m1b7g9ykbzqfn


In [114]:
get_aligned_model_url= f"{BASE_URL}/api/v3/models/{model_id}"

In [115]:
get_aligned_model_response = requests.get(get_aligned_model_url, headers=headers)

print(get_aligned_model_response.text)

{"model_id":"model_ak-31-august-my_vision_alignment__alignment_01m1b7g9ykbzqfn","name":"Ak-31-august-My Vision Alignment  (alignment_01M1B7G9YKBZQFN)","base_model":"qwen2-vl-2b-instruct","alignment_id":"alignment_01M1B7G9YKBZQFN","deployment_status":"READY","created_at":"2026-08-31 06:24:35.870050","updated_at":"2026-08-31 08:47:07.749503"}


**Deploy Aligned Model**

Once alignment completes, you'll be able to deploy the model.


In [116]:
response_data = json.loads(get_aligned_model_response.text)

model_id = response_data["model_id"]

In [117]:
deploy_url = f"{BASE_URL}/api/v3/models/{model_id}/deployment"

In [118]:
deploy_response = requests.post(deploy_url, headers=headers)

print(deploy_response.text)

{"model_id":"model_ak-31-august-my_vision_alignment__alignment_01m1b7g9ykbzqfn"}


**Deploy Status**

Check the deployment status of an aligned model.

In [120]:
response_data = json.loads(deploy_response.text)

model_id = response_data["model_id"]

In [121]:
deploy_status_url = f"https://api.nugen.in/api/v3/models/{model_id}/deployment/status"

In [122]:
response = requests.get(deploy_status_url, headers=headers)

print(response.text)

{"model_id":"model_ak-31-august-my_vision_alignment__alignment_01m1b7g9ykbzqfn","status":"READY","result":{"deployment_status":"DEPLOYED"},"start_time":"2026-08-31T08:48:44.307953","end_time":"2026-08-31T08:48:44.318574"}


**Run Inference**

Once alignment completes, you'll receive an Aligned Model ID.
Use this model for inference.

In [123]:
inference_url = f"{BASE_URL}/api/v3/inference/chat/completions"

The following image_b64 converted image is provided only as an example. Replace it with your own image_b64 image to run inference.

In [124]:
image_b64 ="iVBORw0KGgoAAAANSUhEUgAAAHgAAAAyCAIAAAAYxYiPAAAHS0lEQVR4nO2YW0gU3x/Azzkzo3s1V/CSa0goPURCUVFSalpZD0pFBUpEFCiRPdQ+Jyj0qBK9JEKQtfnkBknaTSV8CyEIfChzH3owg71MM66z4865/B/Ov21z10ulY7/fbz4Py87MmXPOfM453++cgYwxYLHxoF+9gTG2vmOzcm3pza17B8wBpnd6uceAEK5785RShFYfbMbYRrRuJhkeEi4DAIAxNj8/L8vyuswpbjkUClFK06/yJiKRCKUUQsjLxONxWZbn5+f/cZN6qWhK6efPnxVFicVic3NzwWDwy5cvsiyrqgoAWFhY2LFjx/79+xcWFsBqq35lCCEIoUAg0NbWlkgk0l3zM2/fvj116pSu6xhjxlhvb++uXbsqKipisdgfdsBsWAqU0sXFxa6ursLCQkmSzpw5093dffbs2eLi4m3btmmapihKfn5+MBiklFJK2e+CMaaUvn79GiEUjUYZY4SQZB+SxRKJBGPs+vXrVVVV/C5N02ZmZgoLCyORyJLCfzlLRfM/giDU1tbqus4P+bOpqqqqqtfrVRSFMWYYRuJn1t4qn56HDh3y+/3JQ/7La+Y9IYQQQhRFsdvtw8PD/KqqqiUlJemi6TL8hpSNIEOMjkajhJCDBw9KkhSNRnVdz8/PBwBACPk9fD6Koij9zBrXECFEEISnT59OTU1duHCBMSYIAgCA/0ajUVEUeUpACBFCcnJyzp8/PzAwwG/nI5q+LldILX8DYuoBxliSpL6+PoRQY2MjQqijo2PLli3t7e2PHj1yOp3RaJTnJYTQ8+fPh4eH3W63pmmEEJvNdvv2bZvNxlZ7Q+CiR0dHz507RwhJ5rrx8fGurq6PHz+2tLTcuHFDEAQIYXZ2NgCgqanp0qVLqqrm5ORkrBNC+O3bN15Vchj4f4/Hw/9srnRxyTFjTJZlSun79++HhoYeP348NTWVlZV1/Phx8L3rNpttcHCwpaWlqalpZGSkoqIiNzc3fZal1/z/JkXRMIzJycmamhpBEBKJhN1u7+zs7OzsDAQCVVVV+fn527dv1zRtenq6u7sbAHDkyJHs7OxEIpFeLR+2N2/eXLlyhS+1ZEOCIBBCampqHjx4kFwlm0YyiPAuaprmcrnq6upmZmZ6e3s9Hs/CwgJjzDAMxlgkEvF6vbIsNzc3B4NBxtiePXtWDU/pgVLTNABAe3s7vzo2NiaKYkdHB7/q8/mqq6sbGxs/ffrEz4RCIa/XGw6HGWPhcNjr9SZjNK9cURRFUebn59U0QqFQMtNuIuIS6bqux2KxysrKsrKynJyceDxus9kwxjyAgu/RkEfMoaGhYDAoy7LL5WKZ1iaEkEcAxlgoFEII5eXlIYRYyvSHEI6MjGCMfT4fxhhCWFBQMDExcfTo0dLSUj5hwWrbKLvdzlN3egGHw7GWPdFG80M0D9ADAwOSJJ08eRIA4Ha729raEEKpaiCEGGMAAGMsEAg0Nzd7PB7DMLKysjI2QCmNRCI+n+/Dhw+qqu7du/f+/fuCIBw4cGBxcZGXcTgckiQ5HA7uNBwO8yAOACCEAACcTqckSRk9YoxFUbx7925PT48kSYZhJC/xAXa73ZOTk263O+NUMI0foiGE09PT9+7dMwxj69atsix7PJ4V7tR13e/3+3w+AABCSNf1W7du6bqeWkYQhDt37jx8+HB8fLyqqqq+vv7EiROUUofDsW/fvnA4zPMqj9qzs7Mul+vZs2f9/f0ul6uvr6+8vLyurg4AMDEx8fXr14wvNqIoAgCuXbt2+fLl1EwIvouGELpcLrAxnxDWzg/RlNKXL182NDQ0Nzc/efLk8OHDlZWVyZWbCn8YwzAKCgpqa2sBAMutTYQQxri8vHz37t0lJSU7d+4sLi7mOa2+vr61tZUvo5s3bzLGqqurIYQXL1589+5da2trf3//4OCgYRiSJAUCgfLycqfTudxj2O12u93+5zo2kDXGcp5zIpFISUkJ37BgjHmeXJWrV6/29fXxUPPq1Stem2EYdrt9bGwsWUyWZf4VhaVsFDHGiqLYbLbR0VF+5lc3LH/JnuUn0cnNnmEYSzJ1UnRBQUHqFjy1WCINnqBevHiRl5dXWFh47Nix2dlZQgjGmBDi9/srKio0TcMYJzeWyVc09i/egq8AfyRVVYuKisrKyvj3s7W/NsViMa4mWRW/t66u7vTp03xcU2cfpZTvyIeHhxsaGuLx+OLiIqW0p6enuLi4tLRUVVX2jxKd4Xv0ynEmFothjHNzc9eeW1I/OrPvqZ83jxCam5srKipKr42XjEQiHo8HIcQricfjuq6Louhyuf6e7fVa+DXRvw1v5bfVsM3eQP85vyz6D5VlrHCF2tKbW/cOmINJM9pi8/em/xEs0SZhiTYJS7RJWKJNwhJtEpZok7BEm4Ql2iQs0SZhiTYJS7RJWKJNwhJtEpZok7BEm4Ql2iQs0SZhiTYJS7RJWKJNwhJtEpZok7BEm4Ql2iQs0SZhiTYJS7RJ/A+1xXkeN7nnKAAAAABJRU5ErkJggg=="

In [125]:
payload = {
    "model": model_id,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "tell me about this image"
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_b64}"
                    }
                }
            ]
        }
    ],
    "max_tokens": 100,
    "temperature": 1,
    "stream": True,
}

response = requests.post(
    inference_url,
    headers=headers,
    json=payload
)

for line in response.iter_lines():
    if line:
        print(line.decode("utf-8"))

data: {"id": "chatcmpl-vision-295c1d7f-1788166139", "object": "chat.completion.chunk", "created": 1788166139, "model": "295c1d7f-4f35-44f0-b0c1-cc69a1bdfd23", "choices": [{"index": 0, "delta": {"content": ""}, "finish_reason": null}]}
data: {"id": "chatcmpl-vision-295c1d7f-1788166139", "object": "chat.completion.chunk", "created": 1788166139, "model": "295c1d7f-4f35-44f0-b0c1-cc69a1bdfd23", "choices": [{"index": 0, "delta": {"content": "F "}, "finish_reason": null}]}
data: {"id": "chatcmpl-vision-295c1d7f-1788166139", "object": "chat.completion.chunk", "created": 1788166139, "model": "295c1d7f-4f35-44f0-b0c1-cc69a1bdfd23", "choices": [{"index": 0, "delta": {"content": "[ "}, "finish_reason": null}]}
data: {"id": "chatcmpl-vision-295c1d7f-1788166139", "object": "chat.completion.chunk", "created": 1788166139, "model": "295c1d7f-4f35-44f0-b0c1-cc69a1bdfd23", "choices": [{"index": 0, "delta": {"content": ""}, "finish_reason": null}]}
data: {"id": "chatcmpl-vision-295c1d7f-1788166139", "obj

**Explanation**

Vision Alignment enables you to create a Vision Alignment Model using your own image dataset, allowing the model to better understand and respond to domain-specific visual content. Instead of relying solely on the model's general knowledge, alignment adapts the model to your organization's data and use case.

The Vision Alignment workflow in this cookbook consists of the following steps:

1. Upload a vision dataset in JSONL format.
2. Generate benchmark questions using **15% of the uploaded dataset** to evaluate the aligned model.
3. Use the remaining **85% of the dataset** for alignment training.
4. Create an alignment project by selecting an alignment-ready Vision Language Model.
5. Monitor the training progress until the alignment is complete.
6. Perform inference using the newly aligned model.

By following this workflow, you can create a vision model that produces more accurate and context-aware responses for your specific application.

**Conclusion**

In this cookbook, you learned how to build a complete Vision Alignment pipeline using the Nugen API. Starting with a vision dataset, you uploaded the data, generated benchmark questions-answers, created an alignment project, monitored the alignment process, and finally used the aligned model for inference.

This workflow provides a simple and effective way to adapt a Vision Language Model to domain-specific image understanding tasks.